<a href="https://colab.research.google.com/github/Andrii2004/SDXL_finetunning/blob/main/autolabeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

folder_path = '/content/drive/MyDrive/cossack_images'
prefix = 'cartoon_frame'

files = [f for f in os.listdir(folder_path)]
files.sort()

for i, filename in enumerate(files, start=1):
    extension = os.path.splitext(filename)[1]
    new_name = f"{prefix}_{i}{extension}"

    old_file = os.path.join(folder_path, filename)
    new_file = os.path.join(folder_path, new_name)

    os.rename(old_file, new_file)
    print(f"Renamed: {filename} -> {new_name}")

In [ ]:
import requests
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import google.generativeai as genai

In [ ]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-6.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-6.7b").to("cuda")

In [ ]:
def create_label_blip(image_path, save_folder):
  raw_image = Image.open(image_path).convert("RGB")

  question = "Question: What's on this picture? Answer:"
  inputs = processor(raw_image, question, return_tensors="pt").to("cuda")

  out = model.generate(**inputs)
  model_answer = processor.decode(out[0], skip_special_tokens=True).strip()

  file_stem = Path(image_path).stem
  label_path = os.path.join(save_folder, f"{file_stem}.txt")

  with open(label_path, 'w', encoding='utf-8') as f:
      f.write(model_answer)

In [ ]:
def create_label_gemini(api_key, image_path, prompt, save_folder):
    genai.configure(api_key=api_key)

    model = genai.GenerativeModel('gemini-3-pro')

    try:
        img = Image.open(image_path)

        response = model.generate_content([prompt, img])

        file_stem = Path(image_path).stem
        label_path = os.path.join(save_folder, f"{file_stem}.txt")

        with open(label_path, 'w', encoding='utf-8') as f:
            f.write(response.text)

    except Exception as e:
        return f"Exception: {e}"

In [ ]:
from pathlib import Path

MODEL = "Gemini" # Gemini or Blip

images_path = Path('/content/drive/MyDrive/cossack_images')
label_path = Path('/content/drive/MyDrive/cossack_images_labels')

for file in images_path.iterdir():
    if file.is_file():
      if MODEL == "Blip":
        create_label_blip(file, label_path)
      if MODEL == "Gemini":
        MY_API_KEY = "" # Enter your API key
        MY_PROMPT = "Create detailed caption to provided image for SDLX Lora fine-tunning, add token <cossack_cartoon_style> /n No add any additional details or markdowns"
        create_label_gemini(MY_API_KEY, file, MY_PROMPT, label_path)